<a href="https://colab.research.google.com/github/Kongbeng-21/SuperAi/blob/main/notebookc5a51ca2f1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Setup

In [ ]:
!pip install openai faiss-cpu langchain langchain-community langchain-core langchain-text-splitters sentence-transformers tiktoken rank_bm25 -q

In [ ]:
from google.colab import userdata
TYPHOON_API_KEY = userdata.get('TYPHOON_API_KEY')

# Import and Config

In [ ]:
import os
import glob
import json
import re
import time
import pandas as pd
from openai import OpenAI

DATA_PATH = "/content"
KB_PATH = "/content"
OUTPUT_PATH     = "/content/submission.csv"
CHECKPOINT_PATH = "/content/checkpoint.json"

MODEL_NAME           = "typhoon-v2.5-30b-a3b-instruct"
DELAY_BETWEEN_CALLS  = 1.0
MAX_RETRIES          = 3

client = OpenAI(
    api_key=TYPHOON_API_KEY,
    base_url="https://api.opentyphoon.ai/v1"
)
print("✓ Config เสร็จแล้ว")

✓ Config เสร็จแล้ว


# Load Knowledge Base

In [ ]:
import os
import glob
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder # <-- นำเข้า Reranker
import numpy as np

# 1. โหลดและแบ่งข้อความ
documents = []
for filepath in [
    "/content/cu_camps.md",
    "/content/ku_camps.md",
    "/content/kmutt_camps.md",
    "/content/kmitl_camps.md",
    "/content/kmutnb_camps.md"
]:
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    documents.append(Document(page_content=content, metadata={"source": filepath}))

print(f"โหลดได้ {len(documents)} ไฟล์")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print(f"✓ แบ่งเอกสารได้ {len(chunks)} chunks")

# 2. สร้าง AI ค้นหา (Embeddings)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10}) # ดึงมาเผื่อ 10 อัน

# สร้าง BM25 Search
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10 # ดึงมาเผื่อ 10 อัน

# 3. โหลด Reranker Model (ตัวทำคะแนนความแม่นยำ)
print("กำลังโหลดโมเดล Reranker (อาจใช้เวลาสักครู่)...")
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')
print("✓ โหลด Reranker เสร็จสิ้น")

def find_relevant_docs(question, choices, top_k=3):
    """ค้นหาแบบ Hybrid + Reranker """
    query = question + " " + " ".join(choices)

    # 1. ดึงข้อมูลหว่านๆ มาก่อนจากทั้ง Vector และ BM25
    try:
        vec_docs = vector_retriever.invoke(query)
        bm25_docs = bm25_retriever.invoke(query)
    except AttributeError:
        vec_docs = vector_retriever.get_relevant_documents(query)
        bm25_docs = bm25_retriever.get_relevant_documents(query)

    # รวมเอกสารและตัดตัวซ้ำ
    combined_content = list(set([doc.page_content for doc in vec_docs + bm25_docs]))

    if not combined_content:
        return ""

    # 2. ให้ Reranker ให้คะแนนความเป๊ะระหว่าง "คำถาม" กับ "เอกสารแต่ละชิ้น"
    pairs = [[question, doc] for doc in combined_content]
    scores = reranker.predict(pairs)

    # 3. เรียงลำดับจากคะแนนมากไปน้อย แล้วคัดมาแค่ top_k ที่ดีที่สุดจริงๆ
    ranked_indices = np.argsort(scores)[::-1]
    best_docs = [combined_content[i] for i in ranked_indices[:top_k]]

    return "\n\n===\n\n".join(best_docs)

print("✓ Hybrid + Reranker พร้อมใช้งานระดับ Grandmaster!")

โหลดได้ 5 ไฟล์
✓ แบ่งเอกสารได้ 22 chunks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


กำลังโหลดโมเดล Reranker (อาจใช้เวลาสักครู่)...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✓ โหลด Reranker เสร็จสิ้น
✓ Hybrid + Reranker พร้อมใช้งานระดับ Grandmaster!


# Prompt และ LLM

In [ ]:
import re
import time

def ask_typhoon(question, choices, context):
    choices_text = "\n".join(f"{i+1}. {c}" for i, c in enumerate(choices))

    # ใส่ Context เข้าไปให้ AI อ่าน
    prompt = f"""คุณคือ AI ผู้ช่วยค้นหาค่ายและกิจกรรมสำหรับนักเรียนและนักศึกษา
จงตอบคำถามโดยอ้างอิงจากข้อมูลใน Context ด้านล่างนี้เป็นหลัก

Context ข้อมูลกิจกรรม:
{context}

คำถาม: {question}

ตอบแบบละเอียดและเป็นประโยชน์ที่สุด โดยอ้างอิงจาก Context เท่านั้น:"""

    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=4096  # รองรับ Prompt ยาวๆ
            )
            raw = resp.choices[0].message.content.strip()

            # ดึงเฉพาะตัวเลขจากคำตอบ
            for num_str in re.findall(r'\d+', raw):
                num = int(num_str)
                if 1 <= num <= 10:
                    return num
            return 9

        except Exception as e:
            err = str(e)
            if "429" in err or "quota" in err.lower():
                wait = 60 * (attempt + 1)
                print(f"  [RATE LIMIT] รอ {wait} วิ...")
                time.sleep(wait)
            else:
                print(f"  [ERROR] {e}")
                time.sleep(3)
    return 9

print("✓ ask_typhoon updated (แก้ max_tokens=4096)")

✓ ask_typhoon updated (แก้ max_tokens=4096)


# Checkpoint

In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print("✓ ลบ checkpoint แล้ว")
else:
    print("ไม่มี checkpoint")

ไม่มี checkpoint


In [ ]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH) as f:
            data = json.load(f)
        print(f"✓ โหลด checkpoint: {len(data)} rows บันทึกแล้ว")
        return data
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(results, f, ensure_ascii=False)

def save_csv(results, total=100):
    rows = [{"id": i, "answer": results.get(str(i), 9)} for i in range(1, total + 1)]
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_PATH, index=False)
    return df

print("✓ checkpoint functions พร้อมใช้งาน")

✓ checkpoint functions พร้อมใช้งาน


# Pipeline

In [ ]:
def run_pipeline():
    questions_df = pd.read_csv(os.path.join(DATA_PATH, "questions.csv"))
    total = len(questions_df)
    print(f"คำถามทั้งหมด: {total} ข้อ\n")

    results = load_checkpoint()
    already_done = set(results.keys())
    skipped = 0

    for _, row in questions_df.iterrows():
        qid      = str(row["id"])
        question = row["question"]
        choices  = [str(row[f"choice_{i}"]) for i in range(1, 11)]

        if qid in already_done:
            skipped += 1
            continue

        # ใช้ keyword search อย่างเดียว (ไม่ต้องใช้ vectorstore)
        context = find_relevant_docs(question, choices, top_k=5)

        answer = ask_typhoon(question, choices, context)

        if answer == 9:
            alt_context = find_relevant_docs(" ".join(choices), [], top_k=5)
            if alt_context:
                answer = ask_typhoon(question, choices, alt_context)

        results[qid] = answer

        status = "🔴" if answer in [9, 10] else "🟢"
        print(f"{status} Q{int(qid):3d}: ตอบ [{answer}] | {question[:55]}...")

        save_checkpoint(results)

        if int(qid) % 10 == 0:
            save_csv(results, total)
            print(f"บันทึก CSV ชั่วคราว ({qid} ข้อ)")

        time.sleep(DELAY_BETWEEN_CALLS)

    if skipped:
        print(f"\n(ข้ามไป {skipped} ข้อที่ทำแล้ว)")

    submission = save_csv(results, total)
    print(f"\n✓ บันทึกแล้ว: {OUTPUT_PATH}")
    print(f"   {len(submission)} rows")
    print(f"   answer=9: {(submission['answer']==9).sum()} ข้อ")
    print(f"   answer=10: {(submission['answer']==10).sum()} ข้อ")
    print(submission.head(10))
    return submission

print("✓ run_pipeline พร้อมใช้งาน")

✓ run_pipeline พร้อมใช้งาน


In [ ]:
question = "มีค่ายวิศวะที่ไหนบ้างช่วงปิดเทอม"

context = find_relevant_docs(question, [], top_k=3)

print("=== Context ที่ดึงมา ===")
print(context)
print()

resp = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": f"""คุณคือ AI ผู้ช่วยค้นหาค่ายและกิจกรรมสำหรับนักเรียนและนักศึกษา
จงตอบคำถามโดยอ้างอิงจากข้อมูลใน Context ด้านล่างนี้เป็นหลัก

Context:
{context}

คำถาม: {question}

ตอบแบบละเอียดและเป็นประโยชน์ที่สุด:"""}],
    temperature=0.0,
    max_tokens=1000
)

print("=== คำตอบ ===")
print(resp.choices[0].message.content)

=== Context ที่ดึงมา ===
## ค่าย Intania Camp
- **จัดโดย**: นิสิตคณะวิศวกรรมศาสตร์ จุฬาลงกรณ์มหาวิทยาลัย
- **กลุ่มเป้าหมาย**: นักเรียนมัธยมปลายที่สนใจเรียนวิศวะ
- **ช่วงเวลา**: ช่วงปิดเทอมใหญ่ มีนาคม - เมษายน
- **รูปแบบ**: ค่ายแนะแนวการเรียน workshop วิศวกรรม กิจกรรมสันทนาการ
- **ช่องทางติดตาม**: Facebook และ Instagram: Intania Camp

===

## ค่าย Ladkrabang Engineering Camp
- **จัดโดย**: นักศึกษาคณะวิศวกรรมศาสตร์ สจล.
- **กลุ่มเป้าหมาย**: นักเรียนมัธยมปลายที่สนใจวิศวกรรม
- **ช่วงเวลา**: ช่วงปิดเทอมใหญ่ มีนาคม - เมษายน
- **รูปแบบ**: ค่ายแนะแนวการเรียน workshop กิจกรรมสันทนาการ
- **ช่องทางติดตาม**: Facebook และ Instagram คณะวิศวะ สจล.

===

## ค่าย North Bangkok Engineering Camp
- **จัดโดย**: นักศึกษาคณะวิศวกรรมศาสตร์ มจพ.
- **กลุ่มเป้าหมาย**: นักเรียนมัธยมปลายที่สนใจวิศวกรรม
- **ช่วงเวลา**: ช่วงปิดเทอมใหญ่ มีนาคม - เมษายน
- **รูปแบบ**: ค่ายแนะแนวการเรียน workshop วิศวกรรม กิจกรรมสันทนาการ
- **ช่องทางติดตาม**: Facebook และ Instagram คณะวิศวะ มจพ.

=== คำตอบ ===
มีค่ายวิศวกรรมสำหรับนักเรี

# Run

In [ ]:
submission = run_pipeline()

FileNotFoundError: [Errno 2] No such file or directory: '/content/questions.csv'

In [ ]:
import shutil
shutil.copy(OUTPUT_PATH, "/kaggle/working/submission_download.csv")
print("✓ พร้อม download แล้ว")